# Week 43, Day 4: Pattern separation and completion in the hippocampus

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w043/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Expansion and competition

Write kwta(h, k), which returns a binary vector with ones at the k largest entries of h. Then for each input overlap, make 20 pairs of entorhinal patterns and average the overlap of their dentate outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2007)
NE, ND, kE, kD = 200, 2000, 40, 40
W_dg = (r.random((ND, NE)) < 0.1).astype(float)

def kwta(h, k):
    ...

def ec_pair(overlap):
    a = np.zeros(NE); idx = r.choice(NE, kE, replace=False); a[idx] = 1
    b = a.copy(); n = int(round((1 - overlap) * kE))
    b[r.choice(idx, n, replace=False)] = 0
    b[r.choice(np.where(a == 0)[0], n, replace=False)] = 1
    return a, b

def ov(a, b):
    return (a * b).sum() / np.sqrt(a.sum() * b.sum())

out_ov = {}
for o in [0.9, 0.7, 0.5, 0.3]:
    vals = []
    for _ in range(20):
        a, b = ec_pair(o)
        vals.append(ov(kwta(W_dg @ a, kD), kwta(W_dg @ b, kD)))
    out_ov[o] = np.mean(vals)
print({o: round(v, 3) for o, v in out_ov.items()})
plt.plot(list(out_ov), list(out_ov.values()), "o-"); plt.plot([0, 1], [0, 1], "k:")
plt.xlabel("entorhinal overlap"); plt.ylabel("dentate overlap"); plt.show()

In [ ]:
import numpy as np
def test_below():
    "for every input overlap, the output overlap is lower"
    assert all(out_ov[o] < o for o in out_ov)
def test_large():
    "an input overlap of 0.9 falls to below 0.7"
    assert out_ov[0.9] < 0.7

test_below()
test_large()
print("All checks passed")

### Exercise: Recall from 30% of a pattern

Write store(P, a), the covariance rule with zero diagonal, and recall(W, cue, k, steps=10), which repeatedly applies W and keeps the k most active cells. Then test recall from cues with 70% of the active cells removed.

In [ ]:
N = 1000

def store(P, a):
    ...

def recall(W, cue, k, steps=10):
    ...

def patterns(M, a):
    k = int(a * N); P = np.zeros((M, N))
    for m in range(M):
        P[m, r.choice(N, k, replace=False)] = 1
    return P

def run_recall(a, M, n_test=10):
    k = int(a * N); P = patterns(M, a); W = store(P, a); q, qc = [], []
    for m in range(n_test):
        cue = P[m].copy(); on = np.where(cue == 1)[0]
        cue[r.choice(on, int(0.7 * k), replace=False)] = 0
        qc.append(ov(cue, P[m])); q.append(ov(recall(W, cue, k), P[m]))
    return np.mean(q), np.mean(qc)

recall_q = {}
recall_q[(0.05, 50)], cue_q = run_recall(0.05, 50)
print(f"sparse (a = 0.05), 50 patterns: cue overlap {cue_q:.2f}, recalled overlap {recall_q[(0.05, 50)]:.3f}")

In [ ]:
import numpy as np
def test_complete():
    "with 50 stored sparse patterns, recall from a 30% cue has overlap above 0.95"
    assert recall_q[(0.05, 50)] > 0.95
def test_cue():
    "the cue itself has overlap below 0.6, so the network did the work"
    assert cue_q < 0.6

test_complete()
test_cue()
print("All checks passed")

### Exercise: Dense against sparse

Run run_recall for sparse patterns (a = 0.05) with 200 and 400 stored patterns, and dense patterns (a = 0.5) with 50 and 200.

In [ ]:
for a, M in [(0.05, 200), (0.05, 400), (0.5, 50), (0.5, 200)]:
    ...
for key, q in recall_q.items():
    print(f"a = {key[0]}, {key[1]} patterns: recalled overlap {q:.3f}")

In [ ]:
import numpy as np
def test_sparse():
    "sparse coding recalls 400 patterns almost perfectly"
    assert recall_q[(0.05, 400)] > 0.95
def test_dense():
    "dense coding fails at 200 patterns"
    assert recall_q[(0.5, 200)] < 0.9

test_sparse()
test_dense()
print("All checks passed")